Step 1: Mount Drive and Load Files

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [26]:
import os
import pandas as pd
import numpy as np

raw_path = "/content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/data/raw"

partners = pd.read_csv(os.path.join(raw_path, "partners.csv"))
partner_pricing = pd.read_csv(os.path.join(raw_path, "partner_pricing.csv"))
monthly_partner_metrics = pd.read_csv(os.path.join(raw_path, "monthly_partner_metrics.csv"))

print("partners shape:", partners.shape)
print("partner_pricing shape:", partner_pricing.shape)
print("monthly_partner_metrics shape:", monthly_partner_metrics.shape)

partners shape: (5000, 18)
partner_pricing shape: (5000, 14)
monthly_partner_metrics shape: (120000, 14)


Step 2: Inspect Columns

In [27]:
print("partners columns:")
print(partners.columns.tolist())

print("\npartner_pricing columns:")
print(partner_pricing.columns.tolist())

print("\nmonthly_partner_metrics columns:")
print(monthly_partner_metrics.columns.tolist())

partners columns:
['partner_id', 'partner_name', 'partner_type', 'industry_vertical', 'partner_size', 'partner_status', 'country', 'state', 'city', 'region', 'integration_type', 'sales_channel', 'risk_tier', 'kyc_status', 'pci_level', 'onboarding_date', 'relationship_manager_region', 'synthetic_record_flag']

partner_pricing columns:
['assignment_id', 'partner_id', 'pricing_plan_id', 'recommended_pricing_plan_id', 'effective_date', 'expiration_date', 'review_due_date', 'negotiated_bps', 'negotiated_per_txn_fee_usd', 'monthly_minimum_fee_usd', 'exception_flag', 'exception_type', 'approval_status', 'approved_by_role']

monthly_partner_metrics columns:
['partner_id', 'month', 'txn_count', 'payment_volume_usd', 'avg_ticket_usd', 'txn_growth_pct', 'volume_growth_pct', 'chargeback_rate', 'auth_approval_rate', 'active_merchants', 'refunds_usd', 'net_revenue_usd', 'gross_margin_usd', 'gross_margin_rate']


Step 3: Create Clean Working Copies

In [28]:
partner_master = partners.copy()
pricing_clean = partner_pricing.copy()
partner_metrics = monthly_partner_metrics.copy()

Step 4: Standardize Partner Master Columns

In [29]:
partner_master["industry"] = partner_master["industry_vertical"]
partner_master["status"] = partner_master["partner_status"]

partner_master.head()

,partner_id,partner_name,partner_type,industry_vertical,partner_size,partner_status,country,state,city,region,integration_type,sales_channel,risk_tier,kyc_status,pci_level,onboarding_date,relationship_manager_region,synthetic_record_flag,industry,status
0,P000001,NorthStar Commerce 0001,Technology Platform,Travel,SMB,Suspended,US,NY,New York,Northeast,API,Direct,High,Approved,Level 3,2019-06-15,Northeast,1,Travel,Suspended
1,P000002,Ironwood Market 0002,ISV,SaaS,Mid-Market,Pilot,US,VA,Norfolk,South,SDK,Direct,Low,Approved,Level 4,2020-02-13,South,1,SaaS,Pilot
2,P000003,Evergreen Commerce 0003,Technology Platform,Professional Services,SMB,Active,US,CO,Colorado Springs,West,API,Direct,Low,Approved,Level 3,2023-03-12,West,1,Professional Services,Active
3,P000004,Summit Platform 0004,ISO/Reseller,Gaming,SMB,Active,US,GA,Augusta,South,API,Referral,Medium,Approved,Level 4,2020-02-13,South,1,Gaming,Active
4,P000005,Prairie PayTech 0005,Merchant Acquirer,Healthcare,SMB,Active,US,CA,San Diego,West,API,Direct,High,Approved,Level 4,2022-03-09,West,1,Healthcare,Active


Step 5: Standardize Pricing Columns

In [30]:
pricing_clean["pricing_tier"] = pricing_clean["pricing_plan_id"]

pricing_clean["effective_date"] = pd.to_datetime(
    pricing_clean["effective_date"],
    errors="coerce"
)

pricing_clean["review_due_date"] = pd.to_datetime(
    pricing_clean["review_due_date"],
    errors="coerce"
)

pricing_clean["expiration_date"] = pd.to_datetime(
    pricing_clean["expiration_date"],
    errors="coerce"
)

today = pd.Timestamp("2026-06-28")
next_90_days = today + pd.Timedelta(days=90)

pricing_clean["contract_status"] = np.select(
    [
        pricing_clean["approval_status"].eq("Expired"),
        pricing_clean["review_due_date"] < today,
        pricing_clean["review_due_date"].between(today, next_90_days)
    ],
    [
        "Expired",
        "Review Overdue",
        "Review Due Soon"
    ],
    default="Active"
)

pricing_clean[
    [
        "assignment_id",
        "partner_id",
        "pricing_plan_id",
        "pricing_tier",
        "recommended_pricing_plan_id",
        "approval_status",
        "review_due_date",
        "contract_status"
    ]
].head()

,assignment_id,partner_id,pricing_plan_id,pricing_tier,recommended_pricing_plan_id,approval_status,review_due_date,contract_status
0,A000001,P000001,HIGH_RISK,HIGH_RISK,HIGH_RISK,Not Required,2025-04-11,Review Overdue
1,A000002,P000002,PLATFORM_ISV,PLATFORM_ISV,PLATFORM_ISV,Not Required,2025-02-06,Review Overdue
2,A000003,P000003,PLATFORM_ISV,PLATFORM_ISV,PLATFORM_ISV,Not Required,2026-01-04,Review Overdue
3,A000004,P000004,HIGH_RISK,HIGH_RISK,HIGH_RISK,Not Required,2027-02-07,Active
4,A000005,P000005,HIGH_RISK,HIGH_RISK,HIGH_RISK,Not Required,2027-08-27,Active


Step 6: Standardize Monthly Metrics Columns

In [31]:
partner_metrics = partner_metrics.reset_index(drop=True)

partner_metrics["metric_id"] = [
    f"M{str(i + 1).zfill(6)}" for i in range(len(partner_metrics))
]

partner_metrics["transaction_month"] = pd.to_datetime(
    partner_metrics["month"],
    errors="coerce"
)

partner_metrics["transaction_count"] = partner_metrics["txn_count"]
partner_metrics["transaction_volume"] = partner_metrics["payment_volume_usd"]
partner_metrics["revenue"] = partner_metrics["net_revenue_usd"]
partner_metrics["growth_pct"] = partner_metrics["txn_growth_pct"]

partner_metrics[
    [
        "metric_id",
        "partner_id",
        "transaction_month",
        "transaction_count",
        "transaction_volume",
        "revenue",
        "growth_pct"
    ]
].head()

,metric_id,partner_id,transaction_month,transaction_count,transaction_volume,revenue,growth_pct
0,M000001,P000001,2024-01-01,100,10526.09,3121.44,NaN
1,M000002,P000001,2024-02-01,105,11242.13,3129.58,5.00
2,M000003,P000001,2024-03-01,108,12009.93,3138.08,2.86
3,M000004,P000001,2024-04-01,132,13850.19,3160.15,22.22
4,M000005,P000001,2024-05-01,122,14086.19,3161.65,-7.58


Step 7: Check Missing Values

In [32]:
print("Missing values in partner_master:")
print(partner_master.isnull().sum().sort_values(ascending=False))

print("\nMissing values in pricing_clean:")
print(pricing_clean.isnull().sum().sort_values(ascending=False))

print("\nMissing values in partner_metrics:")
print(partner_metrics.isnull().sum().sort_values(ascending=False))

Missing values in partner_master:
partner_id                     0
partner_name                   0
partner_type                   0
industry_vertical              0
partner_size                   0
partner_status                 0
country                        0
state                          0
city                           0
region                         0
integration_type               0
sales_channel                  0
risk_tier                      0
kyc_status                     0
pci_level                      0
onboarding_date                0
relationship_manager_region    0
synthetic_record_flag          0
industry                       0
status                         0
dtype: int64

Missing values in pricing_clean:
expiration_date                5000
approved_by_role               4170
exception_type                 4122
assignment_id                     0
recommended_pricing_plan_id       0
effective_date                    0
pricing_plan_id                   0
partner

Step 8: Validate Primary Keys

In [33]:
duplicate_partner_ids = partner_master["partner_id"].duplicated().sum()
duplicate_assignment_ids = pricing_clean["assignment_id"].duplicated().sum()
duplicate_metric_ids = partner_metrics["metric_id"].duplicated().sum()

duplicate_partner_months = partner_metrics.duplicated(
    subset=["partner_id", "month"]
).sum()

print("Duplicate partner_id values:", duplicate_partner_ids)
print("Duplicate assignment_id values:", duplicate_assignment_ids)
print("Duplicate metric_id values:", duplicate_metric_ids)
print("Duplicate partner-month rows:", duplicate_partner_months)

Duplicate partner_id values: 0
Duplicate assignment_id values: 0
Duplicate metric_id values: 0
Duplicate partner-month rows: 0


Step 9: Validate Foreign Keys

In [34]:
pricing_unmatched = pricing_clean[
    ~pricing_clean["partner_id"].isin(partner_master["partner_id"])
]

metrics_unmatched = partner_metrics[
    ~partner_metrics["partner_id"].isin(partner_master["partner_id"])
]

print("Pricing records without matching partner:", len(pricing_unmatched))
print("Metric records without matching partner:", len(metrics_unmatched))

Pricing records without matching partner: 0
Metric records without matching partner: 0


Step 10: Check Category Values

In [35]:
print("Partner status values:")
print(partner_master["partner_status"].value_counts())

print("\nIndustry vertical values:")
print(partner_master["industry_vertical"].value_counts())

print("\nPartner type values:")
print(partner_master["partner_type"].value_counts())

print("\nPartner size values:")
print(partner_master["partner_size"].value_counts())

print("\nPricing tier values:")
print(pricing_clean["pricing_tier"].value_counts())

print("\nApproval status values:")
print(pricing_clean["approval_status"].value_counts())

print("\nDerived contract status values:")
print(pricing_clean["contract_status"].value_counts())

Partner status values:
partner_status
Active        4013
Pilot          387
Dormant        292
Suspended      223
Terminated      85
Name: count, dtype: int64

Industry vertical values:
industry_vertical
Retail                   721
SaaS                     575
Fintech                  497
Healthcare               466
Logistics                415
Professional Services    405
Hospitality              378
Utilities                345
Travel                   309
Education                261
Nonprofit                254
Gaming                   218
Government               156
Name: count, dtype: int64

Partner type values:
partner_type
ISO/Reseller             927
Technology Platform      834
ISV                      766
Merchant Acquirer        655
Financial Institution    590
Marketplace              508
Gateway                  473
Bank Sponsor             247
Name: count, dtype: int64

Partner size values:
partner_size
SMB           1974
Mid-Market    1603
Enterprise    1006
Strategi

Step 11: Check Numeric Ranges

In [36]:
pricing_clean[
    [
        "negotiated_bps",
        "negotiated_per_txn_fee_usd",
        "monthly_minimum_fee_usd"
    ]
].describe()

,negotiated_bps,negotiated_per_txn_fee_usd,monthly_minimum_fee_usd
count,5000.000000,5000.000000,5000.00000
mean,69.691442,0.063163,1378.41860
std,27.321828,0.023550,1471.19655
min,5.000000,0.013400,0.00000
25%,49.487500,0.045400,99.00000
50%,69.575000,0.059800,499.00000
75%,91.055000,0.081300,1999.00000
max,123.570000,0.113800,4999.00000


In [37]:
partner_metrics[
    [
        "transaction_count",
        "transaction_volume",
        "avg_ticket_usd",
        "growth_pct",
        "volume_growth_pct",
        "chargeback_rate",
        "auth_approval_rate",
        "revenue",
        "gross_margin_usd",
        "gross_margin_rate"
    ]
].describe()

,transaction_count,transaction_volume,avg_ticket_usd,growth_pct,volume_growth_pct,chargeback_rate,auth_approval_rate,revenue,gross_margin_usd,gross_margin_rate
count,1.200000e+05,1.200000e+05,120000.000000,115000.000000,115000.000000,120000.000000,120000.000000,1.200000e+05,1.200000e+05,120000.000000
mean,4.881905e+04,4.011550e+06,121.867150,2.194633,1.922911,0.005826,0.915094,2.537831e+04,1.091384e+04,0.471528
std,1.792433e+05,1.092624e+07,76.398548,11.419830,8.761205,0.004911,0.034864,8.513043e+04,3.442730e+04,0.113436
min,1.000000e+01,1.000000e+03,11.220000,-41.310000,-31.330000,0.000100,0.764890,1.108000e+01,4.030000e+00,0.050000
25%,1.510000e+03,1.560190e+05,67.150000,-5.730000,-4.030000,0.002170,0.892400,1.986380e+03,9.232325e+02,0.406500
50%,5.749000e+03,5.785573e+05,97.560000,1.560000,1.540000,0.004690,0.917660,5.513815e+03,2.446735e+03,0.489900
75%,2.814300e+04,2.769361e+06,159.230000,9.390000,7.420000,0.008020,0.940100,1.801437e+04,8.354802e+03,0.553800
max,1.161502e+07,3.009404e+08,500.850000,65.350000,48.310000,0.027140,0.995000,4.627920e+06,2.419939e+06,0.750000


Step 12: Test Business Question Coverage

Question 1

Show me technology partners in Arizona with transaction growth above 20%.

In [38]:
q1_data = partner_master.merge(
    partner_metrics,
    on="partner_id",
    how="inner"
)

q1_result = q1_data[
    (q1_data["partner_type"].str.contains("Technology", case=False, na=False)) &
    (q1_data["state"] == "AZ") &
    (q1_data["growth_pct"] > 20)
]

q1_result[
    [
        "partner_id",
        "partner_name",
        "partner_type",
        "industry_vertical",
        "state",
        "transaction_month",
        "growth_pct"
    ]
].head(10)

,partner_id,partner_name,partner_type,industry_vertical,state,transaction_month,growth_pct
155,P000007,Clearwater Solutions 0007,Technology Platform,Retail,AZ,2024-12-01,27.44
166,P000007,Clearwater Solutions 0007,Technology Platform,Retail,AZ,2025-11-01,24.59
167,P000007,Clearwater Solutions 0007,Technology Platform,Retail,AZ,2025-12-01,31.87
4036,P000169,Brightpath PayWorks 0169,Technology Platform,Gaming,AZ,2024-05-01,32.07
4047,P000169,Brightpath PayWorks 0169,Technology Platform,Gaming,AZ,2025-04-01,26.28
9057,P000378,RedRock Solutions 0378,Technology Platform,Retail,AZ,2024-10-01,21.95
9069,P000378,RedRock Solutions 0378,Technology Platform,Retail,AZ,2025-10-01,34.85
10188,P000425,Summit Market 0425,Technology Platform,Education,AZ,2025-01-01,27.37
12380,P000516,Riverbend Solutions 0516,Technology Platform,Professional Services,AZ,2025-09-01,32.81
14388,P000600,Cedar Gateway 0600,Technology Platform,Healthcare,AZ,2025-01-01,23.59


In [39]:
print("Technology partners in AZ with growth > 20%:", len(q1_result))

Technology partners in AZ with growth > 20%: 85


Question 2

Which partners are on the Strategic pricing tier?

In [40]:
q2_data = partner_master.merge(
    pricing_clean,
    on="partner_id",
    how="inner"
)

q2_result = q2_data[q2_data["pricing_tier"] == "STRATEGIC"]

q2_result[
    [
        "partner_id",
        "partner_name",
        "pricing_tier",
        "recommended_pricing_plan_id",
        "approval_status"
    ]
].head(10)

,partner_id,partner_name,pricing_tier,recommended_pricing_plan_id,approval_status
20,P000021,Granite Gateway 0021,STRATEGIC,STRATEGIC,Not Required
27,P000028,NorthStar Merchant Services 0028,STRATEGIC,STRATEGIC,Not Required
42,P000043,Prairie PayTech 0043,STRATEGIC,STRATEGIC,Not Required
55,P000056,Cedar PayTech 0056,STRATEGIC,STRATEGIC,Not Required
65,P000066,Copper Financial 0066,STRATEGIC,STRATEGIC,Approved
66,P000067,Evergreen Processing 0067,STRATEGIC,STRATEGIC,Not Required
79,P000080,Granite Market 0080,STRATEGIC,STRATEGIC,Not Required
92,P000093,RedRock Payments 0093,STRATEGIC,STRATEGIC,Not Required
107,P000108,Copper Commerce 0108,STRATEGIC,STRATEGIC,Not Required
110,P000111,Maple Checkout 0111,STRATEGIC,STRATEGIC,Not Required


In [41]:
print("Strategic pricing partners:", len(q2_result))

Strategic pricing partners: 391


Question 3

Which partners have pricing reviews due in the next 90 days?

In [42]:
q3_data = partner_master.merge(
    pricing_clean,
    on="partner_id",
    how="inner"
)

q3_result = q3_data[
    q3_data["review_due_date"].between(today, next_90_days)
]

q3_result[
    [
        "partner_id",
        "partner_name",
        "pricing_tier",
        "approval_status",
        "review_due_date",
        "contract_status"
    ]
].head(10)

,partner_id,partner_name,pricing_tier,approval_status,review_due_date,contract_status
5,P000006,Brightpath PayWorks 0006,PLATFORM_ISV,Not Required,2026-09-08,Review Due Soon
11,P000012,Harbor PayWorks 0012,SCALE,Not Required,2026-08-12,Review Due Soon
28,P000029,Summit Financial 0029,HIGH_RISK,Not Required,2026-09-22,Review Due Soon
58,P000059,Sunrise Commerce 0059,GROWTH,Not Required,2026-08-23,Review Due Soon
65,P000066,Copper Financial 0066,STRATEGIC,Approved,2026-08-25,Review Due Soon
89,P000090,Maple Commerce 0090,HIGH_RISK,Not Required,2026-09-26,Review Due Soon
90,P000091,Ironwood Market 0091,LAUNCH,Not Required,2026-07-26,Review Due Soon
92,P000093,RedRock Payments 0093,STRATEGIC,Not Required,2026-07-17,Review Due Soon
93,P000094,Cedar PayWorks 0094,GROWTH,Not Required,2026-08-01,Review Due Soon
107,P000108,Copper Commerce 0108,STRATEGIC,Not Required,2026-08-15,Review Due Soon


In [43]:
print("Pricing reviews due in next 90 days:", len(q3_result))

Pricing reviews due in next 90 days: 361


Question 4

Which industry has the highest average transaction volume?

In [44]:
q4_data = partner_master.merge(
    partner_metrics,
    on="partner_id",
    how="inner"
)

q4_result = (
    q4_data
    .groupby("industry_vertical")["transaction_volume"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

q4_result

,industry_vertical,transaction_volume
0,Gaming,5.018721e+06
1,Travel,4.521112e+06
2,Fintech,4.449627e+06
3,Utilities,4.153576e+06
4,Logistics,4.149629e+06
5,Healthcare,3.949840e+06
6,Government,3.896428e+06
7,Hospitality,3.887503e+06
8,Retail,3.876467e+06
9,Professional Services,3.814276e+06


Question 5

Show the top 5 partners by revenue.

In [45]:
q5_data = partner_master.merge(
    partner_metrics,
    on="partner_id",
    how="inner"
)

q5_result = (
    q5_data
    .groupby(["partner_id", "partner_name"])["revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)

q5_result

,partner_id,partner_name,revenue
0,P000610,Skyline Payments 0610,64847292.41
1,P002651,RedRock PayTech 2651,28539531.11
2,P003122,Cedar Market 3122,27485493.95
3,P004602,NorthStar Commerce 4602,26392278.10
4,P001422,Brightpath Platform 1422,25564051.57


Question 6

Which active partners are on pricing exceptions?

In [46]:
q6_data = partner_master.merge(
    pricing_clean,
    on="partner_id",
    how="inner"
)

q6_result = q6_data[
    (q6_data["partner_status"] == "Active") &
    (q6_data["exception_flag"] == 1)
]

q6_result[
    [
        "partner_id",
        "partner_name",
        "partner_status",
        "pricing_tier",
        "exception_flag",
        "exception_type",
        "negotiated_bps",
        "approval_status"
    ]
].head(10)

,partner_id,partner_name,partner_status,pricing_tier,exception_flag,exception_type,negotiated_bps,approval_status
19,P000020,Oakstone Gateway 0020,Active,PLATFORM_ISV,1,Discounted bps,29.94,Approved
23,P000024,RedRock PayTech 0024,Active,PLATFORM_ISV,1,Legacy plan extension,37.24,Pending
26,P000027,Prairie Financial 0027,Active,LEGACY_STANDARD,1,Discounted bps,59.12,Rejected
40,P000041,Cedar Merchant Services 0041,Active,LEGACY_STANDARD,1,High-risk waiver,72.08,Pending
43,P000044,Silverline Financial 0044,Active,LEGACY_STANDARD,1,Legacy plan extension,48.50,Approved
45,P000046,NorthStar Checkout 0046,Active,SCALE,1,High-risk waiver,47.83,Approved
50,P000051,Harbor Market 0051,Active,GROWTH,1,High-risk waiver,66.50,Approved
51,P000052,Riverbend Gateway 0052,Active,PROMO_GROWTH,1,Discounted bps,38.70,Expired
60,P000061,Skyline Platform 0061,Active,SCALE,1,Volume threshold waiver,30.06,Expired
61,P000062,Oakstone Financial 0062,Active,SCALE,1,Volume threshold waiver,49.63,Approved


In [47]:
print("Active partners with pricing exceptions:", len(q6_result))

Active partners with pricing exceptions: 681


Step 13: Create Phase 3 Validation Summary

In [48]:
validation_summary = pd.DataFrame({
    "Validation Check": [
        "partner_master row count",
        "pricing_clean row count",
        "partner_metrics row count",
        "Duplicate partner_id in partner_master",
        "Duplicate assignment_id in pricing_clean",
        "Duplicate metric_id in partner_metrics",
        "Duplicate partner-month records in partner_metrics",
        "Pricing records without matching partner",
        "Metric records without matching partner",
        "Technology Platform partners in AZ with growth > 20%",
        "Partners on STRATEGIC pricing tier",
        "Pricing reviews due in next 90 days",
        "Active partners with pricing exceptions"
    ],
    "Result": [
        len(partner_master),
        len(pricing_clean),
        len(partner_metrics),
        duplicate_partner_ids,
        duplicate_assignment_ids,
        duplicate_metric_ids,
        duplicate_partner_months,
        len(pricing_unmatched),
        len(metrics_unmatched),
        len(q1_result),
        len(q2_result),
        len(q3_result),
        len(q6_result)
    ]
})

validation_summary

,Validation Check,Result
0,partner_master row count,5000
1,pricing_clean row count,5000
2,partner_metrics row count,120000
3,Duplicate partner_id in partner_master,0
4,Duplicate assignment_id in pricing_clean,0
5,Duplicate metric_id in partner_metrics,0
6,Duplicate partner-month records in partner_met...,0
7,Pricing records without matching partner,0
8,Metric records without matching partner,0
9,Technology Platform partners in AZ with growth...,85


Step 14: Save Cleaned Files for Phase 4

In [49]:
processed_path = "/content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/data/processed"

os.makedirs(processed_path, exist_ok=True)

partner_master.to_csv(
    os.path.join(processed_path, "partner_master_clean.csv"),
    index=False
)

pricing_clean.to_csv(
    os.path.join(processed_path, "partner_pricing_clean.csv"),
    index=False
)

partner_metrics.to_csv(
    os.path.join(processed_path, "partner_metrics_clean.csv"),
    index=False
)

validation_summary.to_csv(
    os.path.join(processed_path, "phase3_validation_summary.csv"),
    index=False
)

print("Files saved to:", processed_path)

Files saved to: /content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/data/processed


Step 15: Save Everything into One Excel Workbook

In [50]:
excel_output_path = os.path.join(
    processed_path,
    "partnerlens_phase3_validated_dataset.xlsx"
)

with pd.ExcelWriter(excel_output_path, engine="openpyxl") as writer:
    partner_master.to_excel(writer, sheet_name="partner_master", index=False)
    pricing_clean.to_excel(writer, sheet_name="partner_pricing", index=False)
    partner_metrics.to_excel(writer, sheet_name="partner_metrics", index=False)
    validation_summary.to_excel(writer, sheet_name="validation_summary", index=False)
    q1_result.head(100).to_excel(writer, sheet_name="q1_tech_az_growth", index=False)
    q2_result.head(100).to_excel(writer, sheet_name="q2_strategic_pricing", index=False)
    q3_result.head(100).to_excel(writer, sheet_name="q3_review_due_90_days", index=False)
    q4_result.to_excel(writer, sheet_name="q4_volume_by_industry", index=False)
    q5_result.to_excel(writer, sheet_name="q5_top_revenue", index=False)

print("Excel workbook saved to:", excel_output_path)

Excel workbook saved to: /content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/data/processed/partnerlens_phase3_validated_dataset.xlsx
